### Settings and import

In [1]:
import sys, os, pathlib, glob, shutil

# uv pip uninstall spectronaut-runner
# uv pip install git+https://github.com/WeiqiangChen/spectronaut-runner.git

from spectronaut_runner import convert_to_htrms, run_dia_search, run_spectral_library_generation, run_directdia_search

In [2]:
# settings
rawfile_folder = '../data/raw'
htrms_folder = "../data/htrms"

fasta_folder = '../data/fasta'
output_folder = "../data/intermediate/Tims_2_QC_files"
report_scheme_folder = "../data/report"

spectronaut_exec_path= r"C:\Program Files (x86)\Biognosys\Spectronaut205\bin\Spectronaut.exe"
sn_version = "SN205"

# Create missing directories
os.makedirs(output_folder, exist_ok=True)

# get a list of files with tag.
def get_files(folder, tag):
    return [str(f) for f in pathlib.Path(folder).glob(f"*{tag}") ] 

# Find tag= *.psar file from all subfolder and move to current folder. 
def move_files(output_folder, tag=".psar"):
    current_folder = pathlib.Path(output_folder)

    for file in current_folder.rglob('*'):
        if file.is_file() and file.suffix == tag:  # strict suffix match
            dest = current_folder / file.name
            if file != dest:
                shutil.move(str(file), str(dest))
                print(f"Moved: {file.name}")
                
# remove the intermediate folders. 
def remove_subfolders(folder):
    current_folder = pathlib.Path(folder)
    for subfolder in current_folder.iterdir():
        if subfolder.is_dir():
            shutil.rmtree(subfolder)
            print(f"Removed: {subfolder.name}")

fasta_files = get_files(fasta_folder, ".fasta")

report_scheme_files = get_files(report_scheme_folder,".rs")

# convert rawfiles to .htrms

In [3]:
# convert raw to htrms......................... if htrms exists, it will not create again, This can be run after each sample is acquired. 
convert_to_htrms(
    binary= r"C:\Program Files (x86)\Biognosys\HTRMS Converter\HTRMSConverter.exe",
    source=rawfile_folder,
    destination=htrms_folder,
)
htrms_files = get_files(htrms_folder,".htrms")
htrms_files

['..\\data\\htrms\\20260420_HT_N8_AU-749_QCdia_10ng_Inj1_9132.htrms',
 '..\\data\\htrms\\20260420_HT_N8_AU-749_QCdia_10ng_Inj3_9135.htrms']

# 5-step DIA anaylsis (search archive/spectral library, dia analysis)

In [4]:
# Step 1.1 Generate search archive .psar from htrms for each file........... Can be parrallized on many nodes.
search_name = f"generate_search_archive{sn_version}"
psar_files = get_files(output_folder, ".psar")
for htrms_file in htrms_files:
    search_name = f"Step1.1_{os.path.splitext(os.path.basename(htrms_file))[0]}"
    if not (pathlib.Path(output_folder)/ f"{search_name}.psar").exists():
        run_spectral_library_generation(
            spectronaut_exec_path= spectronaut_exec_path,
            output_dir=f"{output_folder}/{search_name}",
            search_name=search_name,
            settings_path=r"../data/settings/SN205_Library_generation_Settings.prop", 
            fasta_paths=fasta_files,
            htrms_paths=[htrms_file],
            search_settings_path = r"../data/settings/SN205_Pulsar_search_Settings.prop",
            skip_library_generation = True,
            extra_cmd_args = ["--pulsarStage", "pulsarStep1"],
        ) 
    print(f"{search_name}.psar is generated or already exist!")
move_files(output_folder, tag=".psar")
remove_subfolders(output_folder)

Step1.1_20260420_HT_N8_AU-749_QCdia_10ng_Inj1_9132.psar is generated or already exist!
Step1.1_20260420_HT_N8_AU-749_QCdia_10ng_Inj3_9135.psar is generated or already exist!
Moved: Step1.1_20260420_HT_N8_AU-749_QCdia_10ng_Inj1_9132.psar
Moved: Step1.1_20260420_HT_N8_AU-749_QCdia_10ng_Inj3_9135.psar
Removed: Step1.1_20260420_HT_N8_AU-749_QCdia_10ng_Inj1_9132
Removed: Step1.1_20260420_HT_N8_AU-749_QCdia_10ng_Inj3_9135


In [5]:
# Step 1.2 generate .qsp files from all .psar files. 
psar_files = get_files(output_folder, ".psar")
search_name = "Step1.2_qsp"
run_spectral_library_generation(
    spectronaut_exec_path = spectronaut_exec_path,
    output_dir = f"{output_folder}/{search_name}",
    search_name = search_name,
    settings_path = r"../data/settings/SN205_Library_generation_Settings.prop", 
    fasta_paths = fasta_files,
    htrms_paths = None,
    search_settings_path = r"../data/settings/SN205_Pulsar_search_Settings.prop",
    search_archive_paths = psar_files,
    skip_library_generation = False,
    extra_cmd_args = ["--pulsarStage", "pulsarStep2"],
) 
move_files(output_folder, tag=".qsp")
remove_subfolders(output_folder)

Moved: Step1.2_qsp.qsp
Removed: Step1.2_qsp


In [6]:
# Step 1.3 generate final .psar files using all .psar and .qsp to get final. .......... Can be parrallized on many nodes.... .psar files in a batch needs their raw files....and the .qsp.
qsp_file = get_files(output_folder, ".qsp")[0]
search_name = "Step1.3_final_psar"
run_spectral_library_generation(
    spectronaut_exec_path = spectronaut_exec_path,
    output_dir = f"{output_folder}/{search_name}",
    search_name = search_name,
    settings_path = r"../data/settings/SN205_Library_generation_Settings.prop", 
    fasta_paths = fasta_files,
    htrms_paths = htrms_files,
    search_settings_path = r"../data/settings/SN205_Pulsar_search_Settings.prop",
    search_archive_paths = psar_files,
    skip_library_generation = True,
    extra_cmd_args = ["--pulsarStage", "pulsarStep3", "--optimizedModels", qsp_file],
) 
move_files(output_folder, tag=".psar")
remove_subfolders(output_folder)

Moved: Step1.3_final_psar.psar
Removed: Step1.3_final_psar


In [7]:
# Step 1.4 generate .kit SL from all FINAL .psar files. 
final_psar_files = get_files(output_folder, "final_psar.psar")

search_name = f"Stepl1.4_SL"
run_spectral_library_generation(
    spectronaut_exec_path= spectronaut_exec_path,
    output_dir=f"{output_folder}/{search_name}",
    search_name=search_name,
    settings_path=r"../data/settings/SN205_Library_generation_Settings.prop", 
    fasta_paths=fasta_files,
    htrms_paths=None,
    search_settings_path = r"../data/settings/SN205_Pulsar_search_Settings.prop",
    skip_library_generation = False,  
    search_archive_paths = final_psar_files,
    library_path = f"{output_folder}/{search_name}.kit"
) 
move_files(output_folder, tag=".kit")
remove_subfolders(output_folder)

Removed: Stepl1.4_SL


In [7]:
# Step 2 DIA analysis using the library
kit_path = get_files(output_folder,".kit")[0]
search_name = "Step2_diaSearch_library_based"
run_dia_search(
    spectronaut_exec_path = spectronaut_exec_path,
    output_dir = f"{output_folder}/{search_name}",
    search_name = search_name,
    settings_path = r"../data/settings/SN205_diaAnalysis_noNorm_noPost.prop", 
    fasta_paths = fasta_files,
    htrms_paths = htrms_files,
    condition_setup_path=r"../data/20260420-test_ConditionSetup.tsv",
    library_path = kit_path,
    report_schema_paths=report_scheme_files,
)

True

# 1-step directDIA+ analysis

In [8]:
search_name = "directDIA_search"
run_directdia_search(
    spectronaut_exec_path= spectronaut_exec_path,
    output_dir=f"{output_folder}/{search_name}",
    search_name=search_name,
    settings_path=r"../data/settings/SN205_directDIA+_noNorm_noPost.prop",
    fasta_paths=fasta_files,
    htrms_paths=htrms_files,
    condition_setup_path=r"../data/20260420-test_ConditionSetup.tsv",
    report_schema_paths=report_scheme_files,
    )

True